In [1]:
import openpyxl
# Data Manipulation and Analysis
import pandas as pd
import numpy as np
# Visualization (for Pearson correlation matrix and plots)
import matplotlib.pyplot as plt
import seaborn as sns
# Machine Learning Data Split & Tuning
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
# Machine Learning Models (Regression)
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression  # This is for Multivariate Linear Regression
import xgboost as xgb
from xgboost import XGBRegressor
# Model Evaluation Metrics (Regression)
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score  # This represents your R-squared score
)
# Feature Importance
import shap
# Optional: Ignore warnings for cleaner outputs
import warnings
warnings.filterwarnings('ignore')
import os
import glob


c:\Users\Phuc\OneDrive\Tài liệu\grad_thesis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# FUNCTIONS

In [13]:
# 3. First Logic: Categorize based on feedback result
def categorize_satisfaction(val):
    '''
    This function lam gi ......
    '''
    if val.startswith('hài lòng'):
        return 'satisfied_ticket'
    elif val.startswith('bình thường'):
        return 'neutral_ticket'
    else:
        # This catches 'vi phạm', 'khong hai long', and any other strange values
        return 'dissatisfied_ticket'

# 4. Second Logic: Categorize based on Method (Phương thức)
def categorize_order(val):
    # Accounting for potential tiny typos in the raw data ('đon' vs 'đơn')
    if val in ['đơn giao hàng', 'Đơn giao hàng']:
        return 'order_delivered_tickets'
    elif val in ['Đơn đến cửa hàng']:
        return 'order_pickup_tickets'
    elif val in ['Bán lẻ']:
        return 'order_onsite_tickets'
    else:
        return 'order_other_tickets'

# Robust cleaning for numeric columns (handling commas, special strings like '11,2 và 15,5')
def clean_numeric(val):
    if pd.isna(val): return np.nan
    if isinstance(val, (int, float)): return float(val)
    # Replace comma with dot for decimal
    s = str(val).replace(',', '.')
    # Extract the first numeric sequence (including decimals)
    import re
    match = re.search(r"[-+]?\d*\.\d+|\d+", s)
    return float(match.group()) if match else np.nan

# NPS

In [ ]:
filepath = r'..\data\raw\nps_raw.xlsx'
print(f"Loading {filepath}...")
nps_df = pd.read_excel(filepath)


In [4]:
# Select only the exact columns we need so we don't carry extra weight
cols_to_keep = ['Ngày tạo/ khảo sát', 'Phân loại kết quả', 'Cửa hàng', 'Phương thức']
nps_df = nps_df[cols_to_keep].dropna(how='all').copy()

# 2. Transform the Date into 'YearMonth' level and rename Store
nps_df['Ngày tạo/ khảo sát'] = pd.to_datetime(nps_df['Ngày tạo/ khảo sát'], errors='coerce')
nps_df['YearMonth'] = nps_df['Ngày tạo/ khảo sát'].dt.to_period('M').astype(str)

# Rename to standard format so the master merge script matches it perfectly
nps_df.rename(columns={'Cửa hàng': 'store_id'}, inplace=True)

# Clean string columns to lowercase for safe matching
nps_df['Phân loại kết quả'] = nps_df['Phân loại kết quả'].astype(str).str.lower().str.strip()
nps_df['Phương thức'] = nps_df['Phương thức'].astype(str).str.lower().str.strip()

In [ ]:
# Create the ticket distribution features
nps_df['NPS_Cat'] = nps_df['Phân loại kết quả'].apply(categorize_satisfaction)
nps_dummies = pd.get_dummies(nps_df['NPS_Cat'], dtype=int)
nps_df = pd.concat([nps_df, nps_dummies], axis=1)

# Ensure all 3 columns exist even if the dataset happens to be missing one entirely
for col in ['satisfied_ticket', 'neutral_ticket', 'dissatisfied_ticket']:
    if col not in nps_df.columns:
        nps_df[col] = 0





In [8]:
nps_df['Order_Cat'] = nps_df['Phương thức'].apply(categorize_order)
order_dummies = pd.get_dummies(nps_df['Order_Cat'], dtype=int)
nps_df = pd.concat([nps_df, order_dummies], axis=1)

for col in ['order_delivered_tickets', 'order_pickup_tickets', 'order_onsite_tickets', 'order_other_tickets']:
    if col not in nps_df.columns:
        nps_df[col] = 0

# 5. Aggregate by Region (store_id) AND Month
agg_funcs = {
    'satisfied_ticket': 'sum',
    'neutral_ticket': 'sum',
    'dissatisfied_ticket': 'sum',
    'order_delivered_tickets': 'sum',
    'order_pickup_tickets': 'sum',
    'order_onsite_tickets': 'sum',
    'order_other_tickets': 'sum'
}

# Apply the aggregation
nps_agg = nps_df.groupby(['store_id', 'YearMonth']).agg(agg_funcs).reset_index()

# 6. Calculate Weighted Average Score
total_tickets = nps_agg['satisfied_ticket'] + nps_agg['neutral_ticket'] + nps_agg['dissatisfied_ticket']

# Avoid division by zero by using np.where (assigns NaN if there are 0 tickets)
nps_agg['nps_weighted_score'] = np.where(
    total_tickets > 0,
    (nps_agg['satisfied_ticket'] * 1 + nps_agg['neutral_ticket'] * 3 + nps_agg['dissatisfied_ticket'] * 5) / total_tickets,
    np.nan
)

In [9]:
# 7. Save to the features folder
out_path = r'..\data\features\nps_aggregated_features.csv'
nps_agg.to_csv(out_path, index=False)
print(f"✅ NPS Feature set created securely in: '{out_path}'")

display(nps_agg.head())

✅ NPS Feature set created securely in: '..\data\features\nps_aggregated_features.csv'


,store_id,YearMonth,satisfied_ticket,neutral_ticket,dissatisfied_ticket,order_delivered_tickets,order_pickup_tickets,order_onsite_tickets,order_other_tickets,nps_weighted_score
0,CPS-AGI-CDO-272LL,2025-10,1,0,0,0,0,0,1,1.000000
1,CPS-AGI-CDO-272LL,2026-01,124,2,0,1,0,0,125,1.031746
2,CPS-AGI-CDO-272LL,2026-03,0,0,1,0,0,0,1,5.000000
3,CPS-AGI-LXG-1393THD,2025-10,121,9,1,1,0,0,130,1.167939
4,CPS-AGI-LXG-1393THD,2026-01,109,0,0,1,0,0,108,1.000000


# SHOP INFO

In [11]:
shop_raw = pd.read_excel(r'..\data\raw\shop_info.xlsx')

cols_to_keep = [
    'ma_shop', 'sizeshop', 'mien', 'tong_dien_tichm2',
    'rong_ngang_m', 'dai_sau_m', 'kv_ban_hangm2',
    'kho_m2', 'via_he_rongm', 'khai_truong_date'
]
shop = shop_raw[cols_to_keep].copy()


In [14]:
num_cols = ['tong_dien_tichm2', 'rong_ngang_m', 'dai_sau_m', 'kv_ban_hangm2', 'kho_m2', 'via_he_rongm']
for col in num_cols:
    shop[col] = shop[col].apply(clean_numeric)

# Map sizeshop: A=4, B=3, C=2, D=1
size_mapping = {'A': 4, 'B': 3, 'C': 2, 'D': 1}
shop['sizeshop'] = shop['sizeshop'].astype(str).str.upper().str.strip().map(size_mapping)

# is_southern: NAM=1, else 0
shop['mien'] = shop['mien'].astype(str).str.upper().str.strip()
shop['is_southern'] = (shop['mien'] == 'NAM').astype(int)
shop.drop(columns=['mien'], inplace=True)

# Convert opening date to datetime
shop['khai_truong_date'] = pd.to_datetime(shop['khai_truong_date'], errors='coerce')

# Rename to match merge key
shop.rename(columns={'ma_shop': 'store_id'}, inplace=True)


In [15]:
# 7. Save to the features folder
out_path = r'..\data\features\shop.csv'
shop.to_csv(out_path, index=False)
print(f"✅ NPS Feature set created securely in: '{out_path}'")



✅ NPS Feature set created securely in: '..\data\features\shop.csv'
